In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

from torch.utils.data import DataLoader, Subset
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# ------------------------------------------------------------
# 1. DEVICE CONFIGURATION
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# CIFAR-10 classes
classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]


# ------------------------------------------------------------
# 2. DATA AUGMENTATION
# ------------------------------------------------------------

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])


# ------------------------------------------------------------
# 3. LOAD CIFAR-10 SUBSET
# ------------------------------------------------------------

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

# Use a smaller subset for faster training
train_subset = Subset(train_dataset, range(5000))
test_subset = Subset(test_dataset, range(1000))

train_loader = DataLoader(
    train_subset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_subset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print("Training samples:", len(train_subset))
print("Testing samples:", len(test_subset))


# ------------------------------------------------------------
# 4. RESIDUAL BLOCK
# ------------------------------------------------------------

class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut connection
        if stride != 1 or in_channels != out_channels:

            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

        else:
            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Residual connection
        out = out + identity

        out = self.relu(out)

        return out


# ------------------------------------------------------------
# 5. PLAIN CNN MODEL
# ------------------------------------------------------------

class PlainCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Linear(128, 10)

    def forward(self, x):

        x = self.features(x)

        x = torch.flatten(x, 1)

        return self.classifier(x)


# ------------------------------------------------------------
# 6. RESIDUAL CNN MODEL
# ------------------------------------------------------------

class ResidualCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            3, 32, kernel_size=3, padding=1, bias=False
        )

        self.bn1 = nn.BatchNorm2d(32)

        self.relu = nn.ReLU(inplace=True)

        self.layer1 = ResidualBlock(32, 64, stride=2)

        self.layer2 = ResidualBlock(64, 128, stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(128, 10)

    def forward(self, x):

        x = self.conv1(x)

        x = self.bn1(x)

        x = self.relu(x)

        x = self.layer1(x)

        x = self.layer2(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        return self.fc(x)


# ------------------------------------------------------------
# 7. TRAINING FUNCTION
# ------------------------------------------------------------

def train_model(model, train_loader, epochs=3, lr=0.001):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    history = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        avg_loss = running_loss / len(train_loader)

        history.append(accuracy)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {avg_loss:.4f} "
            f"Accuracy: {accuracy:.2f}%"
        )

    return history


# ------------------------------------------------------------
# 8. EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_model(model, test_loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    return accuracy


# ------------------------------------------------------------
# 9. TRAIN PLAIN CNN
# ------------------------------------------------------------

print("\n========== PLAIN CNN TRAINING ==========")

plain_model = PlainCNN()

plain_history = train_model(
    plain_model,
    train_loader,
    epochs=3
)

plain_accuracy = evaluate_model(
    plain_model,
    test_loader
)

print("Plain CNN Test Accuracy:", plain_accuracy)


# ------------------------------------------------------------
# 10. TRAIN RESIDUAL CNN
# ------------------------------------------------------------

print("\n========== RESIDUAL CNN TRAINING ==========")

residual_model = ResidualCNN()

residual_history = train_model(
    residual_model,
    train_loader,
    epochs=3
)

residual_accuracy = evaluate_model(
    residual_model,
    test_loader
)

print("Residual CNN Test Accuracy:", residual_accuracy)


# ------------------------------------------------------------
# 11. COMPARE PLAIN CNN AND RESIDUAL CNN
# ------------------------------------------------------------

print("\n========== CNN COMPARISON ==========")

print("Plain CNN Accuracy:", plain_accuracy)

print("Residual CNN Accuracy:", residual_accuracy)

plt.figure(figsize=(8, 5))

plt.plot(plain_history, marker='o', label='Plain CNN')

plt.plot(residual_history, marker='s', label='Residual CNN')

plt.xlabel("Epochs")

plt.ylabel("Training Accuracy (%)")

plt.title("Plain CNN vs Residual CNN")

plt.legend()

plt.grid()

plt.show()


# ------------------------------------------------------------
# 12. PREPARE DATA FOR MOBILENETV3
# ------------------------------------------------------------

weights = MobileNet_V3_Small_Weights.DEFAULT

pretrained_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )

])

transfer_train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=pretrained_transform
)

transfer_test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=pretrained_transform
)

transfer_train_subset = Subset(
    transfer_train_dataset,
    range(5000)
)

transfer_test_subset = Subset(
    transfer_test_dataset,
    range(1000)
)

transfer_train_loader = DataLoader(
    transfer_train_subset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

transfer_test_loader = DataLoader(
    transfer_test_subset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)


# ------------------------------------------------------------
# 13. LOAD PRETRAINED MOBILENETV3-SMALL
# ------------------------------------------------------------

def create_mobilenet():

    model = mobilenet_v3_small(weights=weights)

    # Replace final classification layer
    input_features = model.classifier[3].in_features

    model.classifier[3] = nn.Linear(
        input_features,
        10
    )

    return model


# ------------------------------------------------------------
# 14. FROZEN FEATURE EXTRACTION
# ------------------------------------------------------------

print("\n========== FROZEN FEATURE EXTRACTION ==========")

frozen_model = create_mobilenet()

# Freeze all feature extraction layers
for param in frozen_model.features.parameters():

    param.requires_grad = False

frozen_history = train_model(
    frozen_model,
    transfer_train_loader,
    epochs=3,
    lr=0.001
)

frozen_accuracy = evaluate_model(
    frozen_model,
    transfer_test_loader
)

print("Frozen Feature Extraction Accuracy:", frozen_accuracy)


# ------------------------------------------------------------
# 15. PARTIAL FINE-TUNING
# ------------------------------------------------------------

print("\n========== PARTIAL FINE-TUNING ==========")

finetune_model = create_mobilenet()

# Freeze all feature layers first
for param in finetune_model.features.parameters():

    param.requires_grad = False

# Unfreeze last two feature blocks
for layer in list(finetune_model.features.children())[-2:]:

    for param in layer.parameters():

        param.requires_grad = True

# Train classifier and selected feature layers
finetune_history = train_model(
    finetune_model,
    transfer_train_loader,
    epochs=3,
    lr=0.0001
)

finetune_accuracy = evaluate_model(
    finetune_model,
    transfer_test_loader
)

print("Partial Fine-Tuning Accuracy:", finetune_accuracy)


# ------------------------------------------------------------
# 16. COMPARE TRANSFER LEARNING METHODS
# ------------------------------------------------------------

print("\n========== TRANSFER LEARNING COMPARISON ==========")

print("Frozen Feature Accuracy:", frozen_accuracy)

print("Partial Fine-Tuning Accuracy:", finetune_accuracy)

plt.figure(figsize=(8, 5))

plt.plot(
    frozen_history,
    marker='o',
    label='Frozen Feature Extraction'
)

plt.plot(
    finetune_history,
    marker='s',
    label='Partial Fine-Tuning'
)

plt.xlabel("Epochs")

plt.ylabel("Training Accuracy (%)")

plt.title("Transfer Learning Comparison")

plt.legend()

plt.grid()

plt.show()


# ------------------------------------------------------------
# 17. FEATURE MAP VISUALIZATION
# ------------------------------------------------------------

def visualize_feature_maps(model, image):

    model.eval()

    feature_maps = {}

    def hook_function(module, input, output):

        feature_maps['features'] = output.detach()

    # Capture the output of the first convolution
    hook = model.conv1.register_forward_hook(
        hook_function
    )

    with torch.no_grad():

        model(image.unsqueeze(0).to(device))

    hook.remove()

    maps = feature_maps['features'].cpu()

    plt.figure(figsize=(12, 8))

    for i in range(min(16, maps.shape[1])):

        plt.subplot(4, 4, i + 1)

        plt.imshow(maps[0, i], cmap='viridis')

        plt.axis('off')

        plt.title(f"Map {i+1}")

    plt.suptitle("Feature Maps of Residual CNN")

    plt.tight_layout()

    plt.show()


sample_image, sample_label = test_dataset[0]

visualize_feature_maps(
    residual_model,
    sample_image
)


# ------------------------------------------------------------
# 18. GRAD-CAM IMPLEMENTATION
# ------------------------------------------------------------

class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model

        self.target_layer = target_layer

        self.activations = None

        self.gradients = None

        self.target_layer.register_forward_hook(
            self.save_activation
        )

        self.target_layer.register_full_backward_hook(
            self.save_gradient
        )

    def save_activation(self, module, input, output):

        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):

        self.gradients = grad_output[0]

    def generate(self, image, class_index=None):

        self.model.eval()

        self.model.zero_grad(set_to_none=True)

        image = image.unsqueeze(0).to(device)

        output = self.model(image)

        if class_index is None:

            class_index = output.argmax(dim=1).item()

        score = output[0, class_index]

        score.backward()

        # Calculate weights
        weights = self.gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        # Weighted feature maps
        cam = (weights * self.activations).sum(dim=1)

        cam = torch.relu(cam)

        # Normalize
        cam = cam - cam.min()

        cam = cam / (cam.max() + 1e-8)

        cam = cam.unsqueeze(1)

        cam = torch.nn.functional.interpolate(
            cam,
            size=(224, 224),
            mode='bilinear',
            align_corners=False
        )

        return cam.squeeze().detach().cpu().numpy(), class_index


# ------------------------------------------------------------
# 19. GENERATE GRAD-CAM
# ------------------------------------------------------------

gradcam = GradCAM(
    finetune_model,
    finetune_model.features[-1]
)

sample_image, sample_label = transfer_test_dataset[0]

cam, predicted_class = gradcam.generate(sample_image)

# Convert normalized image for display
mean = np.array(weights.transforms().mean)

std = np.array(weights.transforms().std)

image_np = sample_image.permute(1, 2, 0).numpy()

image_np = image_np * std + mean

image_np = np.clip(image_np, 0, 1)

# Display Grad-CAM
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)

plt.imshow(image_np)

plt.title("Original Image")

plt.axis('off')

plt.subplot(1, 3, 2)

plt.imshow(cam, cmap='jet')

plt.title("Grad-CAM Heatmap")

plt.axis('off')

plt.subplot(1, 3, 3)

plt.imshow(image_np)

plt.imshow(cam, cmap='jet', alpha=0.5)

plt.title("Grad-CAM Overlay")

plt.axis('off')

plt.suptitle(
    f"Actual: {classes[sample_label]} | "
    f"Predicted: {classes[predicted_class]}"
)

plt.tight_layout()

plt.show()


# ------------------------------------------------------------
# 20. FINAL RESULTS
# ------------------------------------------------------------

print("\n======================================")
print("          FINAL RESULTS")
print("======================================")

print(f"Plain CNN Accuracy: {plain_accuracy:.2f}%")

print(f"Residual CNN Accuracy: {residual_accuracy:.2f}%")

print(f"Frozen Feature Accuracy: {frozen_accuracy:.2f}%")

print(f"Partial Fine-Tuning Accuracy: {finetune_accuracy:.2f}%")

print("\nProgram Executed Successfully!")